## Данные экспериментов

Эксперименты были проведены на чистых/грязных данных с двумя ассистентами - gemma:2b (local) и gpt-oss-20B (cloud). \
Цель экспериментов - сравнить, насколько использование усовершенствованных промптов влияет на качество ответов ассистентов в разных конфигурациях. 

### Метрики и их описание

* **latency** - время ответа ассистента на один поставленный вопрос. Начало отсчета - время получения вопроса и контекста ассистентом, конец - когда ответ дошел до системы. Измеряется в милисекундах. **Меньше = лучше**
* **token_count** - количество затраченных токенов на один вопрос (входные и выходные токены суммируются). **Меньше = лучше**
* **bert_score** - автоматическая метрика для оценки качества сгенерированного текста. Она использует нейросети для измерения смыслового сходства между эталонным и сгенерированным текстами, в отличие от алгоритмов, требующих точного совпадения слов. Измеряется от 0 до 1. **Больше = лучше** 
* **jaccard_distance** - метрика для измерения степени различия между наборами слов в текстах. Измеряется от 0 до 1. **Меньше = лучше** 
* **rouge_l_f1** - метрика схожести текстов на основе Самой Длинной Общей Последовательности слов (LCS - Longest Common Subsequence) Измеряется от 0 до 1. **Больше = лучше** 
* **hallucination_rate** - метрика для измерения уровня галлюцинаций (выдуманных фактов) с помощью моделей естественного языкового вывода (NLI - Natural Language Inference). Составляется как разность между показателем противоречивости ответа относительно контекста и семантического следствия ответа из контекста. Принимает любые значения, но чаще всего меняется в диапазоне от -5 до 5, где на интервале от -1 до 1 модель-оценщик не уверена в уровне галлюцинаций в ответе. Значения ниже -1 - низкий уровень галлюцинаций, выше 1 - высокий. **Меньше = лучше**
* **numeric_accuracy** - метрика для проверки точного совпадения числе в ответе и в эталоне. Изменяется от нуля до единицы. Если в эталоне нет или не обнаружено чисел, метрика принимает значение "1". **Больше = лучше**
* **overlap** - метрика перекрытия n-грамм. Подсчитывает процент совпадения элементов. **Больше = лучше**

In [1]:
import sys
sys.path.append('/workspace/chat-prep-etl/src/testing_system/cmd')

In [2]:
from notebooks.analysis import aggregate_experiment_metrics
FOLDER_1 = "/workspace/chat-prep-etl/experiment_results_gemma"
FOLDER_2 = "/workspace/chat-prep-etl/experiment_results_gpt"
df = aggregate_experiment_metrics(FOLDER_1, FOLDER_2)
df

,Модель,Промпт,Данные,BERT_SCORE,HALLUCINATION_RATE,JACCARD,LATENCY,NUMERIC_ACCURACY,OVERLAP,ROUGE_L_F1,TOKEN_COUNT
0,experiment_results_gemma,baseline,clean,0.6459,-0.1533,0.9197,2391.2050,0.6056,0.1941,0.0281,952.1000
1,experiment_results_gemma,baseline,raw,0.6442,-0.3985,0.9282,2379.0295,0.6079,0.1933,0.0188,981.9667
2,experiment_results_gemma,evolution,clean,0.6345,-0.5353,0.9349,2840.0744,0.5922,0.1597,0.0184,1081.4333
3,experiment_results_gemma,evolution,raw,0.6346,-0.0970,0.9381,4290.0616,0.5781,0.1583,0.0193,1046.7091
4,experiment_results_gpt,baseline,clean,0.6326,-1.5487,0.9239,24147.4828,0.6357,0.2608,0.0430,4781.5333
5,experiment_results_gpt,baseline,raw,0.6264,-1.4882,0.9314,10651.9766,0.6394,0.2483,0.0567,2731.7000
6,experiment_results_gpt,evolution,clean,0.6323,-1.4948,0.9257,12558.4486,0.6295,0.2580,0.0366,3099.1500
7,experiment_results_gpt,evolution,raw,0.6338,-1.8475,0.9228,13470.5286,0.6276,0.2632,0.0441,3197.2500


Анализ результатов экспериментов позволяет сделать ряд выводов относительно влияния типа промпта (базовый baseline против улучшенного evolution) на качество и эффективность работы двух моделей — Gemma и GPT. В целом, эволюционный промпт не продемонстрировал однозначного превосходства над базовым. Его воздействие сильно зависит от конкретной модели и характера данных (чистые или сырые).

Рассмотрим сначала модель Gemma. Здесь переход от baseline к evolution привёл к ухудшению практически всех ключевых метрик. На чистых данных время ответа (latency) выросло примерно на 19%, потребление токенов — на 13,5%. Смысловая точность (BERT_score) снизилась с 0,6459 до 0,6345. Метрики, чувствительные к структуре текста — overlap, ROUGE_L_F1, Jaccard (чем меньше, тем лучше для Jaccard, но здесь он вырос, что хуже) — также ухудшились. Единственное заметное улучшение — снижение уровня галлюцинаций (hallucination_rate) с -0,1533 до -0,5353, то есть ответы стали реже противоречить контексту. На сырых данных ситуация ещё показательнее: latency взлетела почти вдвое (с 2379 до 4290 мс), токенов стало больше, а галлюцинации не только не снизились, но и выросли (с -0,3985 до -0,0970). Числовая точность (numeric_accuracy) упала. Таким образом, для Gemma эволюционный промпт оказался скорее регрессом: он замедлил работу, увеличил расходы на токены и в большинстве случаев ухудшил качество генерации, за редким исключением в виде снижения галлюцинаций на идеальных данных.

Перейдём к модели GPT. Здесь картина получилась более противоречивой, но и более интересной с точки зрения оптимизации промптинга. На чистых данных evolution промпт дал впечатляющее сокращение времени ответа — более чем в два раза (с 24147 мс до 12558 мс) и уменьшение потребления токенов почти на 35%. При этом качественные метрики изменились незначительно и в основном в худшую сторону: BERT_score практически не изменился, но hallicunation_rate чуть вырос (стал менее отрицательным), Jaccard и overlap ухудшились на доли процента, ROUGE_L_F1 снизился с 0,0430 до 0,0366. То есть мы получили значительный выигрыш в скорости и экономии токенов ценой очень небольшого падения смысловой точности и связности — такой компромисс может быть оправдан во многих сценариях. На сырых данных эффект обратный: эволюционный промпт неожиданно увеличил latency (с 10652 до 13471 мс) и расход токенов. Зато качественные показатели улучшились: BERT_score вырос с 0,6264 до 0,6338, а главное — уровень галлюцинаций резко снизился с -1,4882 до -1,8475 (значение намного ниже -1, что говорит об очень низком уровне выдумок). Jaccard уменьшился (лучше), overlap увеличился (лучше). Таким образом, на грязных данных эволюционный промпт для GPT повысил достоверность и точность ответов, пусть и ценой замедления.

Что же это означает для практики промптинга? Эволюция — не панацея. Для Gemma она оказалась вредна почти по всем параметрам, вероятно, из-за того, что усложнённые инструкции перегружают модель меньшего размера или плохо сочетаются с её архитектурой. Для GPT эффект разнонаправлен: на чистых данных evolution полезен для ускорения и экономии, на грязных — для борьбы с галлюцинациями и повышения робастности. Следовательно, выбор между baseline и evolution должен опираться на приоритеты: если важна скорость и низкая стоимость — на чистых данных с GPT имеет смысл использовать evolution; если важна достоверность при работе с зашумлёнными данными — тоже evolution, но придётся мириться с возросшей задержкой. Для Gemma же безоговорочно лучше оставаться на базовом промпте. Универсального «лучшего» промпта не существует — его эффективность глубоко привязана к модели и к качеству входных данных.